In [0]:
# Configuration: source and target schemas, plus the tables to load.
# Edit table_names to load a subset (e.g. ["product", "vendor"]).

source_schema = "adventure_works.bronze"
target_schema = "adventure_works.silver"

# Silver target table name -> actual bronze table name
# (bronze tables were created with different names than the exercise uses)
bronze_name_map = {
    "product": "product",
    "purchase_order_detail": "stg_purchase_order_detail",
    "purchase_order_header": "stg_purchase_order_header",
    "sales_order_detail": "salesorderdetail",
    "sales_territory": "salesterritory",
    "vendor": "dim_vendor",
}

table_names = list(bronze_name_map.keys())

In [0]:
# Full load: read each bronze table, trim whitespace on string columns,
# and overwrite the silver table. Silver is built from bronze (not raw CSVs).
from functools import reduce
from pyspark.sql import functions as F

for name in table_names:
    bronze_table = f"{source_schema}.{bronze_name_map[name]}"
    silver_table = f"{target_schema}.{name}"

    df = spark.table(bronze_table)

    # Light cleaning: trim surrounding whitespace on all string columns.
    # Bronze is already typed, so no casting is needed.
    string_cols = [c for c, dtype in df.dtypes if dtype == "string"]
    if string_cols:
        df = reduce(
            lambda acc, c: acc.withColumn(c, F.trim(F.col(c))),
            string_cols,
            df,
        )

    rows_written = df.count()
    (
        df.write.mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(silver_table)
    )
    print(f"{name}: wrote {rows_written} rows to {silver_table}")

In [0]:
# Verification: bronze and silver row counts must match for every table.
from pyspark.sql import functions as F
from pyspark.sql.types import BooleanType, LongType, StringType, StructField, StructType

results = []
for name in table_names:
    bronze_rows = spark.table(f"{source_schema}.{bronze_name_map[name]}").count()
    silver_rows = spark.table(f"{target_schema}.{name}").count()
    results.append((name, bronze_rows, silver_rows, bronze_rows == silver_rows))

verification_df = spark.createDataFrame(
    results,
    schema=StructType([
        StructField("table", StringType(), False),
        StructField("bronze_rows", LongType(), False),
        StructField("silver_rows", LongType(), False),
        StructField("counts_match", BooleanType(), False),
    ]),
)
display(verification_df)

mismatches = verification_df.filter(F.col("counts_match") == False).count()
assert mismatches == 0, f"{mismatches} table(s) have bronze/silver row count mismatches"
print("Success: all bronze and silver row counts match.")